Senior thesis abstract:

The Multiband Imaging Survey for High-Alpha PlanetS (MISHAPS) aims to identify transiting hot Jupiters around Main Sequence stars in the Galactic Bulge prior to the launch of the Nancy Grace Roman Space Telescope in May 2027. Given the sensitivity of the transit method to large planets orbiting small stars, accurate stellar radii are essential for identifying viable planetary hosts. Using DECam photometry, we calibrated magnitudes and colors to the PanSTARRS system, where color-diameter relationships are well established. To address the effects of significant interstellar dust along the line of sight to the Bulge, we determined extinction and reddening values using statistical fits to giant stars and Red Clump stars in the Red Giant Branch. After dereddening, we estimated stellar radii for over 50,000 stars and validated these estimates against a PanSTARRS isochrone. The resulting radius distribution matched the general shape of the isochrone despite limitations in proper motion filtering. Our extinction and reddening estimates follow expected trends with wavelength, and the fractional residuals showed a typical scatter of $\sigma_{r} \approx 0.38$ for the r-band and $\sigma_{i} \approx 0.28$ for the i-band, with increasing dispersion toward faint magnitudes due to photometric uncertainties. These results support the feasibility of deriving accurate stellar parameters in crowded, dusty fields and provide a foundation for future hot Jupiter transit searches in the Galactic Bulge.

The relevant code for this project it below. It also isn't all of it since there's a lot, and I didn't include the data files. So, don't run it because it won't work. Thank you.

In [4]:
# Imports
import numpy as np  # Array operations
import matplotlib.pyplot as plt  # Plotting
from matplotlib.colors import LinearSegmentedColormap, to_rgba_array # For the color scheme
import matplotlib.gridspec as gridspec
import pandas as pd  # Data handling
#import pyp2options  # Custom library --> Calibrations
from astropy.coordinates import SkyCoord  # Astronomical coordinate handling
import astropy.units as u  # Physical units
import scipy.optimize as spopt  # Optimization algorithms
import sys

In [8]:
# Defined Parameters 
field_name = 'MISHAPS_F1_DITH0'  # Field identifier
chip_id = 'N10'  # Chip identifier
magnitude_band = 'r'  # Band used for magnitude measurement
color_band_1 = 'r'  # First band for color (e.g. r-z)
color_band_2 = 'z'  # Second band for color
field_chip_label = f"{field_name}_{chip_id}"  # Combined label for file references

# CMD (Color-Magnitude Diagram) Parameters
default_color_cut = 1.06
magnitude_cut = 17.3
color_range_min = 0.0
color_range_max = 2.0
msto_shift_value = 3.5  # Main Sequence Turn-Off shift

# Adjust defaults for specific color combinations
if color_band_2 == "z" and color_band_1 == "r":
    # default_color_cut = 0.46
    default_color_cut = 0.6
    color_range_max = 1.0
    msto_shift_value = 3.0

# Enable interactive plots if enough command-line arguments are provided
interactive_mode = False
if len(sys.argv) >= 6:
    interactive_mode = True

# Optionally override color and magnitude cuts
if len(sys.argv) >= 7:
    default_color_cut = float(sys.argv[6])
    print("Color cut set to:", default_color_cut)

if len(sys.argv) >= 8:
    magnitude_cut = float(sys.argv[7])
    print("Magnitude cut set to:", magnitude_cut)

print("Using magnitude and color bands:", magnitude_band, color_band_1, color_band_2)

# Optional positional filtering via RA/Dec limits
ra_min, ra_max = -1, -1
dec_min, dec_max = -1, -1
apply_position_cut = False

if len(sys.argv) >= 9:
    ra_min = float(sys.argv[8])
    ra_max = float(sys.argv[9])
    dec_min = float(sys.argv[10])
    dec_max = float(sys.argv[11])
    apply_position_cut = True

Using magnitude and color bands: r r z


. . .

In [ ]:
# Construct Color-Magnitude Diagram (CMD)
### Compute color index and magnitude
color_index = filtered_catalog[:, c1_col_index] - filtered_catalog[:, c2_col_index]
magnitude_values_r = filtered_catalog[:, mag_col_index] # r-band
magnitude_values_z = filtered_catalog[:, c2_col_index] # z-band

In [10]:
# Red Clump Gaussian 
def cgauss(x, amplitude, mean, sigma):
    x = np.float64(x)
    return amplitude/(np.sqrt(2*np.pi*sigma**2))*np.exp(-0.5*(x-mean)**2/sigma**2)

In [ ]:
# Red Clump Star Selection (r-band)
# Filter stars that are likely within the red clump
clump_candidates = giant_candidates[np.abs(giant_candidates[:, mag_col_index] - par_0[2]) < 2 * par_0[4]]

clump_colors = clump_candidates[:, c1_col_index] - clump_candidates[:, c2_col_index]
clump_magnitudes = clump_candidates[:, mag_col_index]

print(f"Selected {len(clump_candidates)} red clump candidates in the r-band")

# For the color distribution fit of stars in the red clump
# Create a histogram of colors
hist_counts_clump, bin_edges_clump = np.histogram(clump_colors, range = (default_color_cut, default_color_cut + 1.5), bins=60)
bin_centers_clump = 0.5 * (bin_edges_clump[:-1] + bin_edges_clump[1:])

# Select only the bins that aren't empty, apply that to the histogram and bin centers
mask_1 = (hist_counts_clump>0)
hist_counts_m1 = hist_counts_clump[mask_1]
bin_centers_m1 = bin_centers_clump[mask_1]

# Initial guess for Gaussian parameters: amplitude, mean, sigma
initial_guess_1 = [par_0[3] + 0.0, default_color_cut + 0.2, 0.2]
initial_guess_3 = [70.0, 0.75, 0.2]

par_1, cov_1 = spopt.curve_fit(cgauss, bin_centers_m1, hist_counts_m1, p0 = initial_guess_3, sigma=np.sqrt(hist_counts_m1))
print(f"cgauss Model Fit Results:\nAmplitude = {par_1[0]:.2f}\n  Mean = {par_1[1]:.2f}\n Sigma = {par_1[2]:.2f}")

amplitude, mean, sigma = par_1

In [13]:
# Plot Red Clump Giant Star Candidates on CMD for r-mag
clump_fig = plt.figure(figsize = (8, 6))
plt.tight_layout()

plt.scatter(color_index, magnitude_values_r, s = 1, color = '#125A56', alpha = 0.1, label = 'All Stars')
plt.scatter(clump_colors, clump_magnitudes, s = 1, color = '#A01813', alpha = 0.3, label = ' Red Clump Candidates')

plt.xlim(-1,3)
plt.gca().invert_yaxis()

plt.title("Giant Stars in Red Clump Candidate Selection (r-band)")
plt.xlabel(f"{color_band_1} - {color_band_2} [Color]")
plt.ylabel(f"{magnitude_band}-band [Magnitude]")

plt.grid(True)
plt.legend(loc='best')

plt.show()

NameError: name 'color_index' is not defined

<Figure size 800x600 with 0 Axes>